In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
 C    0.879477   -0.347438    0.000000
 C    0.000000    0.675064    0.000000
 C   -0.879440   -0.347453    0.000000
 H    1.829361   -0.828858    0.000000
 H   -0.000385    1.776911    0.000000
 H   -1.829195   -0.829095    0.000000'''
molecule = 'C3H3_an'
spin = 0
charge =  -1


In [ ]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 4 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [4]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [ ]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

output file: ./C3H3_an.txt
orbital  energy   occ
0     -10.872685688416341    2.0
1     -10.671433481526192    2.0
2     -10.671254053116067    2.0
3     -0.6961170365556815    2.0
4     -0.4178497770390106    2.0
5     -0.3984830991701227    2.0
6     -0.2967733997970312    2.0
7     -0.15883956036481606    2.0
8     -0.07809325932559494    2.0
9     0.08140341225421666    2.0
10     0.23900944275585634    2.0
11     0.664081623330719    0.0
12     0.7409501651928719    0.0
13     0.9196833120603833    0.0
14     0.9642606954942453    0.0
15     0.9790350212619534    0.0
16     1.3499989418295932    0.0
17     1.568972425160214    0.0
Homo_index: 10, Lumo_index: 11


WARN: System has nonzero charge -1; the dipole moment is origin-dependent.
Location of origin: [0. 0. 0.]


In [6]:
n_elec= 2
n_orb = 3 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([9,11,12])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-114.31364431258432),
 np.float64(-0.5646213643360767),
 FCIvector([[-9.69076840e-01, -2.71240789e-05, -1.51865917e-01],
            [-2.71240789e-05,  7.07618536e-02, -1.03379064e-05],
            [-1.51865917e-01, -1.03379064e-05,  9.87741050e-02]]),
 NPArrayWithTag([[ 2.40120710e-03, -7.81551785e-01,  6.16272543e-01,
                  -1.18031022e-01, -1.41191585e-01,  8.87251842e-02,
                  -1.74161538e-02, -2.65729003e-02,  2.56899742e-02,
                  -1.05702999e-01,  0.00000000e+00,  0.00000000e+00,
                   0.00000000e+00,  1.17137621e-01,  1.19959362e-01,
                   5.79577620e-02,  2.76781512e-02, -3.15631862e-02],
                 [-4.67656294e-03, -1.81779809e-02,  1.25415782e-02,
                   2.87357877e-01,  4.29312022e-01, -2.58881492e-01,
                   8.22320479e-02,  1.01143748e-01, -1.36666210e-01,
                   5.55388574e-01,  0.00000000e+00,  0.00000000e+00,
                   0.00000000e+00, -7.694136

In [7]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0]          [0]          -0.969076839695765
[0]          [1]          -0.000027124078904
[0]          [2]          -0.151865916724300
[1]          [0]          -0.000027124078904
[1]          [1]          0.070761853630745
[1]          [2]          -0.000010337906382
[2]          [0]          -0.151865916724300
[2]          [1]          -0.000010337906382
[2]          [2]          0.098774105036749
